In [ ]:
# Installs core dependencies for Ultralytics YOLO pose inference and video utilities
!pip -q install -U ultralytics opencv-python numpy pandas ipywidgets ipyevents pillow

In [ ]:
# Imports libraries for data engineering, video processing, and model inference
import os
import io
import time
import json
import math
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image

# Selects a YOLO pose model to balance inference latency and keypoint quality for video scale processing
from ultralytics import YOLO

import ipywidgets as widgets

from ipyevents import Event
from IPython.display import display, Video, HTML

# Defines run_cmd to encapsulate reusable logic within the video analytics pipeline
def run_cmd(cmd):
    """Run a shell command safely and print stdout/stderr on failure."""
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return out
    except subprocess.CalledProcessError as e:
        print("Command failed:", " ".join(cmd))
        print(e.output)
        raise


In [ ]:
# Imports libraries for data engineering, video processing, and model inference










import os
import shlex
import subprocess

BUCKET = "msads-mba-capstone-team-2"

AM_VIDEO = "data/amateur/YouTube_3-0-Pickleball-Match.mp4"
PRO_VIDEO = "data/pro/YouTube_MEN-S-PRO-GOLD-2024-US-Open-Pickleball.mp4"

RUN_DIR = "runs_ready_score"
os.makedirs(RUN_DIR, exist_ok=True)

LOCAL_AM_VIDEO = os.path.join(RUN_DIR, "amateur_source.mp4")

LOCAL_PRO_VIDEO = os.path.join(RUN_DIR, "pro_source.mp4")

AM_CLIP = os.path.join(RUN_DIR, "amateur_clip1.mp4")
AM_CLIP_2 = os.path.join(RUN_DIR, "amateur_clip2.mp4")
AM_CLIP_3 = os.path.join(RUN_DIR, "amateur_clip3.mp4")

PRO_CLIP = os.path.join(RUN_DIR, "pro_clip1.mp4")
PRO_CLIP_2 = os.path.join(RUN_DIR, "pro_clip2.mp4")
PRO_CLIP_3 = os.path.join(RUN_DIR, "pro_clip3.mp4")

AM_START = "00:06:40"
AM_START_2 = "00:07:35"
AM_START_3 = "00:13:38"

PRO_START = "00:33:06"
PRO_START_2 = "00:19:18"
PRO_START_3 = "00:25:00"

DUR_S = 10
TARGET_FPS = 30
TARGET_W = 1280


# Defines run_cmd to encapsulate reusable logic within the video analytics pipeline
def run_cmd(cmd):
"""Run a shell command safely and print stdout/stderr on failure."""
try:
    out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
    return out
except subprocess.CalledProcessError as e:










    print("Command failed:", " ".join(shlex.quote(c) for c in cmd))
    print(e.output)
    raise

# Defines _as_gs_uri to encapsulate reusable logic within the video analytics pipeline
def _as_gs_uri(p: str) -> str:
"""Convert relative bucket path -> gs://BUCKET/... (leave gs:// as-is)."""
if p.startswith("gs://"):
    return p
p2 = p.lstrip("/") # normalize
return f"gs://{BUCKET}/{p2}"

# Defines ensure_local_video to encapsulate reusable logic within the video analytics pipeline
def ensure_local_video(src_path: str, local_path: str) -> str:
"""
Ensures a playable local MP4 exists at local_path.
- If src_path exists locally, use it.
- Else try copying from GCS using BUCKET + provided relative path.

- If that fails, try a fallback using the same filename under Data/... folders
(useful if objects actually live in Data/Amateur Videos/ or Data/
Professional Videos/).
"""
if os.path.exists(src_path):
    return src_path
if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
    print(f"Local file already exists: {local_path}")
    return local_path
candidates = []


fname = os.path.basename(src_path)
candidates.append(f"gs://{BUCKET}/Data/Amateur Videos/{fname}")
candidates.append(f"gs://{BUCKET}/Data/Professional Videos/{fname}")
last_err = None
for gs_uri in candidates:
    try:
        print(f"Copying from GCS -> local: {gs_uri} -> {local_path}")
        run_cmd(["gsutil", "-m", "cp", gs_uri, local_path])










        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        return local_path
    except Exception as e:
        last_err = e
        continue

raise FileNotFoundError(
    "Could not find/copy the video from any of these GCS locations:\n"
    + "\n".join(f" - {c}" for c in candidates)
    + "\n\nIf gsutil is authenticated, one of those paths should work. "
    "If not, run: `gcloud auth login` (or use the notebook’s auth flow)."
) from last_err

AM_VIDEO_LOCAL = ensure_local_video(AM_VIDEO, LOCAL_AM_VIDEO)
PRO_VIDEO_LOCAL = ensure_local_video(PRO_VIDEO, LOCAL_PRO_VIDEO)


print("AM_VIDEO_LOCAL:", AM_VIDEO_LOCAL)
print("PRO_VIDEO_LOCAL:", PRO_VIDEO_LOCAL)
AM_VIDEO_LOCAL: data/amateur/YouTube_3-0-Pickleball-Match.mp4
PRO_VIDEO_LOCAL: data/pro/YouTube_MEN-S-PRO-GOLD-2024-US-Open-Pickleball.mp4


In [ ]:
# Imports libraries for data engineering, video processing, and model inference
import os

# Defines a helper that standardizes a fixed duration clip with ffmpeg for consistent inference
def extract_10s_clip(src, start_ts, out_path, duration=DUR_S, fps=TARGET_FPS, width=TARGET_W):
    """
    Extract a normalized H.264 MP4 clip for inference.
    start_ts can be "HH:MM:SS" (recommended) or seconds as float/int.
    """
    if os.path.exists(out_path):
        os.remove(out_path)

    cmd = [
        "ffmpeg", "-hide_banner", "-y",
        "-ss", str(start_ts),
        "-i", src,
        "-t", str(float(duration)),
        "-vf", f"fps={int(fps)},scale={int(width)}:-2",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        out_path











    ]
    run_cmd(cmd)

extract_10s_clip(AM_VIDEO_LOCAL, AM_START, AM_CLIP)
extract_10s_clip(AM_VIDEO_LOCAL, AM_START_2, AM_CLIP_2)
extract_10s_clip(AM_VIDEO_LOCAL, AM_START_3, AM_CLIP_3)

extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START, PRO_CLIP)
extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START_2, PRO_CLIP_2)
extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START_3, PRO_CLIP_3)

print("Wrote clips:")
print(" ", AM_CLIP)
print(" ", AM_CLIP_2)

print(" ", AM_CLIP_3)
print(" ", PRO_CLIP)
print(" ", PRO_CLIP_2)
print(" ", PRO_CLIP_3)

Wrote clips:
runs_ready_score/amateur_clip1.mp4
runs_ready_score/amateur_clip2.mp4
runs_ready_score/amateur_clip3.mp4
runs_ready_score/pro_clip1.mp4
runs_ready_score/pro_clip2.mp4
runs_ready_score/pro_clip3.mp4

In [ ]:
# Defines _dist to encapsulate reusable logic within the video analytics pipeline
        def _dist(a, b):
return float(np.linalg.norm(a - b))

# Defines _angle to encapsulate reusable logic within the video analytics pipeline
        def _angle(a, b, c):
"""Angle ABC in degrees (at b)."""
ba = a - b
bc = c - b
nba = np.linalg.norm(ba)
nbc = np.linalg.norm(bc)
if nba < 1e-6 or nbc < 1e-6:
    return np.nan
cosang = float(np.clip(np.dot(ba, bc) / (nba * nbc), -1.0, 1.0))
return float(np.degrees(np.arccos(cosang)))

# Defines _valid_xy to encapsulate reusable logic within the video analytics pipeline
        def _valid_xy(p):












return (p is not None) and np.isfinite(p).all() and (p[0] > 0) and (p[1] > 0)
# Defines ready_score_from_kpts to encapsulate reusable logic within the video analytics pipeline
        def ready_score_from_kpts(kpts_xy):
"""
ReadyScore in [0,100], HIGHER is better.
100 = excellent ready posture; 0 = poor readiness.
"""
k = np.asarray(kpts_xy, dtype=float)
if k.shape != (17, 2):
    return np.nan


L_EL, R_EL = 7, 8
L_WR, R_WR = 9, 10
L_HI, R_HI = 11, 12
L_KN, R_KN = 13, 14
L_AN, R_AN = 15, 16

req = [L_SH, R_SH, L_HI, R_HI, L_KN, R_KN, L_AN, R_AN]
if not all(_valid_xy(k[i]) for i in req):
    return np.nan

sh_mid = (k[L_SH] + k[R_SH]) / 2.0
hi_mid = (k[L_HI] + k[R_HI]) / 2.0
shoulder_w = _dist(k[L_SH], k[R_SH])
_ = max(shoulder_w, _dist(sh_mid, hi_mid), 1.0) # scale placeholder (kept for clarity)
lk = _angle(k[L_HI], k[L_KN], k[L_AN])
rk = _angle(k[R_HI], k[R_KN], k[R_AN])
knee_ang = np.nanmean([lk, rk])
if not np.isfinite(knee_ang):
    return np.nan

knee_target = 145.0
knee_pen = abs(knee_ang - knee_target) / 45.0

stance = _dist(k[L_AN], k[R_AN]) / max(shoulder_w, 1.0)
if stance < 1.1:
    stance_pen = (1.1 - stance) / 0.6
elif stance > 2.0:
    stance_pen = (stance - 2.0) / 1.0
else:









    stance_pen = 0.0

torso_vec = sh_mid - hi_mid
n = np.linalg.norm(torso_vec)
if n < 1e-6:
    return np.nan
v = torso_vec / n
vert = np.array([0.0, -1.0])
lean_deg = float(np.degrees(np.arccos(np.clip(np.dot(v, vert), -1.0, 1.0))))
if lean_deg < 10:
    lean_pen = (10 - lean_deg) / 15.0
elif lean_deg > 25:
    lean_pen = (lean_deg - 25) / 25.0
else:
    lean_pen = 0.0


hand_pen = 0.5
wrists_ok = _valid_xy(k[L_WR]) and _valid_xy(k[R_WR])
if wrists_ok:
    wr_mid = (k[L_WR] + k[R_WR]) / 2.0
    denom = max((hi_mid[1] - sh_mid[1]), 1.0)
    r = float((wr_mid[1] - sh_mid[1]) / denom)
    if r < 0.2:
        vpen = (0.2 - r) / 0.2
    elif r > 0.85:
        vpen = (r - 0.85) / 0.35
    else:
        vpen = 0.0
    hpen = abs(float(wr_mid[0] - sh_mid[0])) / max(shoulder_w, 1.0)
    hpen = max(0.0, (hpen - 0.7) / 0.8)
    hand_pen = 0.65 * vpen + 0.35 * hpen

raw_bad = (1.20 * knee_pen) + (0.90 * stance_pen) + (0.70 * lean_pen) + (0.
            80 * hand_pen)
bad_score = 100.0 * (raw_bad / 4.0)
readiness = 100.0 - float(np.clip(bad_score, 0.0, 100.0))
return float(np.clip(readiness, 0.0, 100.0))

In [ ]:
if "MODEL_WEIGHTS" not in globals():
    MODEL_WEIGHTS = "yolov8n-pose.pt"
if "CONF" not in globals():










    CONF = 0.25

# Selects a YOLO pose model to balance inference latency and keypoint quality for video scale processing
model = YOLO(MODEL_WEIGHTS)
print("Loaded model:", MODEL_WEIGHTS, "| CONF:", CONF)

Loaded model: yolov8n-pose.pt | CONF: 0.25


In [ ]:
# Imports libraries for data engineering, video processing, and model inference

        import re
        import numpy as np
        import cv2

        SELECT_T_SEC = 1.0

        TARGET_DESC = {
"AM1": "pink shirt",
"AM2": "pink shirt",
"AM3": "pink shirt",
"PR1": "blue shirt and white pants",
"PR2": "blue shirt and white pants",
"PR3": "blue shirt and white pants",

        }

        HSV_RANGES = {
"red": [((0, 80, 60), (10, 255, 255)), ((170, 80, 60), (179, 255, 255))],
"orange":[((10, 80, 60), (22, 255, 255))],
"yellow":[((22, 80, 60), (35, 255, 255))],
"green": [((35, 60, 50), (85, 255, 255))],
"blue": [((90, 60, 50), (130, 255, 255))],
"purple":[((130, 50, 50), (165, 255, 255))],
"pink": [((145, 50, 60), (170, 255, 255))],
"white": [((0, 0, 200), (179, 60, 255))],
"black": [((0, 0, 0), (179, 255, 50))],
"gray": [((0, 0, 50), (179, 40, 200))],
"navy": [((100, 80, 20), (135, 255, 120))], # rough
        }












        REGION_SYNONYMS = {
"hat": "hat",
"cap": "hat",
"shirt": "shirt",
"top": "shirt",
"tee": "shirt",
"tshirt": "shirt",
"shorts": "shorts",
"pants": "shorts", # treat as lower body region
"trousers": "shorts",
        }

        REGION_WEIGHTS = {
"hat": 0.55,

"shirt": 0.45,
"shorts": 0.35,
None: 0.40, # if user says just "blue" with no region
        }

        MIN_PER_CONSTRAINT = {
"hat": 0.015,
"shirt": 0.020,
"shorts": 0.020,
None: 0.020,
        }

        MIN_TOTAL_SCORE = 0.03


# Defines _extract_frame_at_time to encapsulate reusable logic within the video analytics pipeline
        def _extract_frame_at_time(video_path, t_sec=1.0):
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")
cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, float(t_sec)) * 1000.0)
ok, frame_bgr = cap.read()
cap.release()

if not ok or frame_bgr is None:
    raise RuntimeError(f"Could not read frame at t={t_sec}s from {video_path}")
return frame_bgr
# Defines _predict_person_boxes_on_frame to encapsulate reusable logic within the video analytics pipeline
        def _predict_person_boxes_on_frame(model, frame_bgr, conf=0.25):
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)










r = model.predict(frame_rgb, conf=conf, verbose=False)[0]
if r.boxes is None or len(r.boxes) == 0:
    return np.zeros((0, 4), dtype=int), frame_rgb
boxes = r.boxes.xyxy.cpu().numpy().astype(int)
return boxes, frame_rgb

# Defines region_crop_from_bbox to encapsulate reusable logic within the video analytics pipeline
        def region_crop_from_bbox(img_rgb, bbox_xyxy, region=None):
"""
region heuristic within person bbox:
hat:  top 25%
shirt: middle 45%
shorts: bottom 45%
None: entire bbox
"""
x1, y1, x2, y2 = map(int, bbox_xyxy)
x1 = max(0, x1); y1 = max(0, y1)

x2 = min(img_rgb.shape[1]-1, x2); y2 = min(img_rgb.shape[0]-1, y2)
if x2 <= x1 or y2 <= y1:
    return None

h = y2 - y1
if region == "hat":
    yy1, yy2 = y1, y1 + int(0.25 * h)
elif region == "shirt":
    yy1, yy2 = y1 + int(0.20 * h), y1 + int(0.65 * h)
elif region == "shorts":
    yy1, yy2 = y1 + int(0.55 * h), y2
else:
    yy1, yy2 = y1, y2

yy1 = max(y1, yy1)
yy2 = min(y2, yy2)
if yy2 <= yy1:
    return None

return img_rgb[yy1:yy2, x1:x2].copy()

# Defines color_fraction_hsv to encapsulate reusable logic within the video analytics pipeline
        def color_fraction_hsv(img_rgb_crop, color_name):
"""Fraction of pixels matching the target color (HSV threshold)."""
if img_rgb_crop is None or img_rgb_crop.size == 0:

    return 0.0
hsv = cv2.cvtColor(img_rgb_crop, cv2.COLOR_RGB2HSV)
masks = []
for (lo, hi) in HSV_RANGES[color_name]:
    lo = np.array(lo, dtype=np.uint8)
    hi = np.array(hi, dtype=np.uint8)
    masks.append(cv2.inRange(hsv, lo, hi))










mask = masks[0]
for m in masks[1:]:
    mask = cv2.bitwise_or(mask, m)
return float(mask.mean() / 255.0)

# Defines parse_multi_desc to encapsulate reusable logic within the video analytics pipeline
        def parse_multi_desc(desc: str):
"""
Parse multiple constraints from:
"blue hat and red shirt" -> [("blue","hat"),("red","shirt")]
"white shirt, black shorts" -> ...
"blue" -> [("blue", None)]
"""
d = desc.lower().strip()
chunks = re.split(r"\s*(?:and|&|\+|,|;)\s*", d)
constraints = []
for ch in chunks:

    toks = re.split(r"\s+", ch.strip())
    color = None
    region = None
    for t in toks:
        if t in HSV_RANGES:
        color = t
        if t in REGION_SYNONYMS:
        region = REGION_SYNONYMS[t]
    if color is not None:
        constraints.append((color, region))

if len(constraints) == 0:
    raise ValueError(
        f"Could not parse any (color, region) from '{desc}'. "
        f"Known colors: {list(HSV_RANGES.keys())}. Regions: {sorted(set(REGION_SYNONYMS.keys()))}"
    )
return constraints

# Defines score_person_for_constraints to encapsulate reusable logic within the video analytics pipeline
        def score_person_for_constraints(frame_rgb, bbox_xyxy, constraints):
"""

Returns:
total_score (weighted sum, normalized),
per_constraint list of dicts.
"""
per = []
total = 0.0
weight_sum = 0.0

for (color, region) in constraints:










    crop = region_crop_from_bbox(frame_rgb, bbox_xyxy, region=region)
    frac = color_fraction_hsv(crop, color)

    w = float(REGION_WEIGHTS.get(region, REGION_WEIGHTS.get(None, 0.4)))
    weight_sum += w
    total += w * frac

    per.append({
        "color": color,
        "region": region,
        "frac": float(frac),
        "weight": float(w),
        "min_required": float(MIN_PER_CONSTRAINT.get(region, 0.02)),
    })

if weight_sum > 1e-9:

    total = total / weight_sum

return float(total), per

# Defines select_target_bbox_by_multi_desc to encapsulate reusable logic within the video analytics pipeline
        def select_target_bbox_by_multi_desc(video_path, model, desc, conf=0.25, t_sec=1.0):
constraints = parse_multi_desc(desc)
frame_bgr = _extract_frame_at_time(video_path, t_sec=t_sec)
boxes, frame_rgb = _predict_person_boxes_on_frame(model, frame_bgr, conf=conf)
if boxes.shape[0] == 0:
    raise RuntimeError(f"No persons detected for {video_path} at t={t_sec}s")
best = {"idx": None, "score": -1.0, "per": None}
all_debug = []
for i, b in enumerate(boxes):

    total, per = score_person_for_constraints(frame_rgb, b, constraints)

    ok = True
    for p in per:
        if p["frac"] < p["min_required"]:
        ok = False
        break

    all_debug.append({"i": i, "total": total, "ok": ok, "per": per})

    if ok and total > best["score"]:
        best = {"idx": int(i), "score": float(total), "per": per}











if best["idx"] is None or best["score"] < MIN_TOTAL_SCORE:
    top = sorted(all_debug, key=lambda x: x["total"], reverse=True)[:3]
    msg = (
        f"Could not confidently match '{desc}' at t={t_sec}s.\n"
        f"Best passing score: {best['score']:.4f} (min total {MIN_TOTAL_SCORE}).\n"
        "Top candidates (even if failing per-constraint mins):\n"
    )
    for cand in top:
        msg += f" - idx {cand['i']}: total={cand['total']:.4f}, ok={cand['ok']}, per={cand['per']}\n"
    msg += (
        "\nTips:\n"
        "- Try a different SELECT_T_SEC (e.g., 2.0–4.0)\n"
        "- Use larger/clearer region like 'red shirt' instead of 'blue hat'\n"
        "- Lower MIN_PER_CONSTRAINT or MIN_TOTAL_SCORE slightly if lighting is tough\n"
    )
    raise RuntimeError(msg)
debug = {
    "desc": desc,
    "constraints": constraints,
    "best_index": best["idx"],
    "best_score": best["score"],
    "best_per": best["per"],
    "t_sec": float(t_sec),
    "num_people": int(len(boxes)),
    "all_debug": all_debug, # can comment out if too verbose
}
return boxes[best["idx"]].astype(float), debug










        PRO_TARGET_BBOX_2, dbg_pr2 = select_target_bbox_by_multi_desc(PRO_CLIP_2, model, TARGET_DESC["PR2"], conf=CONF, t_sec=SELECT_T_SEC)
        PRO_TARGET_BBOX_3, dbg_pr3 = select_target_bbox_by_multi_desc(PRO_CLIP_3, model, TARGET_DESC["PR3"], conf=CONF, t_sec=SELECT_T_SEC)
        print("Selected target bboxes (xyxy):")
        print(" AM1:", AM_TARGET_BBOX)
        print(" AM2:", AM_TARGET_BBOX_2)
        print(" AM3:", AM_TARGET_BBOX_3)
        print(" PR1:", PRO_TARGET_BBOX)
        print(" PR2:", PRO_TARGET_BBOX_2)
        print(" PR3:", PRO_TARGET_BBOX_3)

        print("\nBest-match debug:")
        print(" AM1:", {"best_index": dbg_am1["best_index"], "best_score": dbg_am1["best_score"], "best_per": dbg_am1["best_per"]})
        print(" PR1:", {"best_index": dbg_pr1["best_index"], "best_score": dbg_pr1["best_score"], "best_per": dbg_pr1["best_per"]})
        Selected target bboxes (xyxy):
            AM1: [     717       28       776      163]
            AM2: [     782       11       854      138]
            AM3: [     774        3       835      142]
            PR1: [     463      177       539      348]
            PR2: [    1059      352      1176      716]
            PR3: [     646      162       721      295]
        Best-match debug:
            AM1: {'best_index': 1, 'best_score': 0.24689265536723165, 'best_per':
        [{'color': 'pink', 'region': 'shirt', 'frac': 0.24689265536723165, 'weight':
        0.45, 'min_required': 0.02}]}
            PR1: {'best_index': 2, 'best_score': 0.20325316131237184, 'best_per':
        [{'color': 'blue', 'region': 'shirt', 'frac': 0.31681476418318527, 'weight':
        0.45, 'min_required': 0.02}, {'color': 'white', 'region': 'shorts', 'frac':
        0.05724538619275461, 'weight': 0.35, 'min_required': 0.02}]}

        run_cmd([“gsutil”,“-m”,“cp”,“-r”,“01_yolo_pose_baseline.ipynb”,“runs/”,“gs://…/”])

        import matplotlib.pyplot as plt
        import numpy as np
        import cv2

# Defines draw_all_players_and_target to encapsulate reusable logic within the video analytics pipeline
        def draw_all_players_and_target(video_path, model, target_bbox, t_sec=1.0, conf=0.25, title=""):
frame_bgr = _extract_frame_at_time(video_path, t_sec=t_sec)









frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


for i, b in enumerate(boxes):
    x1, y1, x2, y2 = map(int, b)
    cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (255, 255, 0), 2) # yellow
    cv2.putText(frame_rgb, f"{i}", (x1, max(0, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.
            LINE_AA)
if target_bbox is not None:

    x1, y1, x2, y2 = map(int, target_bbox)
    cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (0, 255, 0), 4) # green
    cv2.putText(frame_rgb, "TARGET", (x1, max(0, y1 - 28)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)

plt.figure(figsize=(14, 7))
plt.imshow(frame_rgb)
plt.title(title + f" (t={t_sec:.1f}s, detected={len(boxes)})")
plt.axis("off")
plt.show()

        draw_all_players_and_target(AM_CLIP, model, AM_TARGET_BBOX, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM1 — {TARGET_DESC['AM1']}")
        draw_all_players_and_target(AM_CLIP_2, model, AM_TARGET_BBOX_2, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM2 — {TARGET_DESC['AM2']}")
        draw_all_players_and_target(AM_CLIP_3, model, AM_TARGET_BBOX_3, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM3 — {TARGET_DESC['AM3']}")
        draw_all_players_and_target(PRO_CLIP, model, PRO_TARGET_BBOX, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR1 — {TARGET_DESC['PR1']}")
        draw_all_players_and_target(PRO_CLIP_2, model, PRO_TARGET_BBOX_2, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR2 — {TARGET_DESC['PR2']}")
        draw_all_players_and_target(PRO_CLIP_3, model, PRO_TARGET_BBOX_3, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR3 — {TARGET_DESC['PR3']}")

In [ ]:
# Defines _iou_xyxy to encapsulate reusable logic within the video analytics pipeline
        def _iou_xyxy(a, b):
ax1, ay1, ax2, ay2 = map(float, a)










bx1, by1, bx2, by2 = map(float, b)
ix1, iy1 = max(ax1, bx1), max(ay1, by1)
ix2, iy2 = min(ax2, bx2), min(ay2, by2)
iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
inter = iw * ih
area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
union = area_a + area_b - inter
return inter / union if union > 1e-9 else 0.0

# Defines summarize_video_ready_score_locked to encapsulate reusable logic within the video analytics pipeline
        def summarize_video_ready_score_locked(
video_path,
model,
init_bbox_xyxy,
conf=0.25,
tracker_cfg="bytetrack.yaml",

init_search_frames=45,
reacquire_iou_thresh=0.25,
        ):
scores = []
target_id = None
last_bbox = None
init_bbox = np.array(init_bbox_xyxy, dtype=float)
frame_idx = 0

for r in model.track(
source=video_path,
conf=conf,
stream=True,
persist=True,
tracker=tracker_cfg,
verbose=False,
):
frame_idx += 1

if r.boxes is None or len(r.boxes) == 0 or r.keypoints is None:
    continue

if getattr(r.boxes, "id", None) is None:

    boxes = r.boxes.xyxy.cpu().numpy().astype(float)
    if boxes.shape[0] == 0:
        continue
    ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
    i = int(np.argmax(ious))
    if ious[i] < 0.05:









        continue
    kpts = r.keypoints.xy[i].cpu().numpy()
    s = ready_score_from_kpts(kpts)
    if np.isfinite(s):
        scores.append(float(s))
    continue

boxes = r.boxes.xyxy.cpu().numpy().astype(float)
ids = r.boxes.id.cpu().numpy().astype(int)

if target_id is None and frame_idx <= init_search_frames:
    ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
    best = int(np.argmax(ious))
    if ious[best] >= 0.10:
        target_id = int(ids[best])
        last_bbox = boxes[best].copy()

idxs = np.where(ids == target_id)[0] if target_id is not None else np.
            array([], dtype=int)
if target_id is not None and len(idxs) == 0 and last_bbox is not None:
    ious = np.array([_iou_xyxy(last_bbox, b) for b in boxes], dtype=float)
    best = int(np.argmax(ious))
    if ious[best] >= reacquire_iou_thresh:
        target_id = int(ids[best])
        idxs = np.array([best], dtype=int)
if len(idxs) == 0:
    continue

i = int(idxs[0])

last_bbox = boxes[i].copy()
kpts = r.keypoints.xy[i].cpu().numpy()
if kpts.shape[0] < 17:
    continue

s = ready_score_from_kpts(kpts)
if np.isfinite(s):
    scores.append(float(s))

scores = np.array(scores, dtype=float)
return {
"target_id": int(target_id) if target_id is not None else None,










"frames_scored": int(len(scores)),
"mean": float(scores.mean()) if len(scores) else None,
"p10": float(np.percentile(scores, 10)) if len(scores) else None,
"p50": float(np.percentile(scores, 50)) if len(scores) else None,
"p90": float(np.percentile(scores, 90)) if len(scores) else None,
}

In [ ]:
am_summary = summarize_video_ready_score_locked(AM_CLIP, model, AM_TARGET_BBOX, conf=CONF)
pro_summary = summarize_video_ready_score_locked(PRO_CLIP, model, PRO_TARGET_BBOX, conf=CONF)
print("AMATEUR:", am_summary)
print("PRO: ", pro_summary)
AMATEUR: {'target_id': 2, 'frames_scored': 290, 'mean': 59.77885066317961,
'p10': 44.57815985148762, 'p50': 61.60942816634097, 'p90': 73.87396034756188}

PRO:   {'target_id': 80, 'frames_scored': 205, 'mean': 70.10498923059207,
'p10': 53.48829459813247, 'p50': 71.77249328983204, 'p90': 83.18279446004516}
frames_scored How many frames produced a valid score This is a data quality check, not a perfor-
mance metric

mean Average readiness across the clip This is your primary scalar metric
p10 Bottom-end readiness Captures worst posture moments

p50 (median) Typical posture during the clip Often more robust than the mean
p90 Best posture moments Shows peak form capability

In [ ]:
# Imports libraries for data engineering, video processing, and model inference
        def write_overlay_raw_locked(
                video_in,
                video_out_raw,
                model,
                init_bbox_xyxy,
                conf=0.25,
                label="",

                tracker_cfg="bytetrack.yaml",
                init_search_frames=45,
                reacquire_iou_thresh=0.25,
        ):
                cap = cv2.VideoCapture(video_in)
                if not cap.isOpened():
                raise RuntimeError(f"Could not open video: {video_in}")











                fps = cap.get(cv2.CAP_PROP_FPS) or 30
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                cap.release()

                if os.path.exists(video_out_raw):
                os.remove(video_out_raw)

                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(video_out_raw, fourcc, fps, (w, h))

                init_bbox = np.array(init_bbox_xyxy, dtype=float)
                target_id = None
                last_bbox = None
                frame_idx = 0


                for r in model.track(
                source=video_in,
                conf=conf,
                stream=True,
                persist=True,
                tracker=tracker_cfg,
                verbose=False,
                ):
                frame_idx += 1
                frame = r.plot() # draws skeletons/boxes as Ultralytics sees them

                score = None
                tgt_bbox = None

                if r.boxes is not None and len(r.boxes) > 0 and r.keypoints is not None:
                    boxes = r.boxes.xyxy.cpu().numpy().astype(float)

                    if getattr(r.boxes, "id", None) is not None:
                        ids = r.boxes.id.cpu().numpy().astype(int)

                        if target_id is None and frame_idx <= init_search_frames:
                        ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
                        best = int(np.argmax(ious))
                        if ious[best] >= 0.10:
                            target_id = int(ids[best])
                            last_bbox = boxes[best].copy()

                        idxs = np.where(ids == target_id)[0] if target_id is not None else np.array([], dtype=int)










                        if target_id is not None and len(idxs) == 0 and last_bbox is not None:
                        ious = np.array([_iou_xyxy(last_bbox, b) for b in boxes], dtype=float)
                        best = int(np.argmax(ious))
                        if ious[best] >= reacquire_iou_thresh:
                            target_id = int(ids[best])
                            idxs = np.array([best], dtype=int)
                        if len(idxs) > 0:
                        i = int(idxs[0])
                        last_bbox = boxes[i].copy()
                        tgt_bbox = last_bbox.copy()
                        kpts = r.keypoints.xy[i].cpu().numpy()
                        if kpts.shape[0] >= 17:
                            score = ready_score_from_kpts(kpts)

                    else:
                        ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
                        i = int(np.argmax(ious))
                        if ious[i] >= 0.05:
                        tgt_bbox = boxes[i].copy()
                        kpts = r.keypoints.xy[i].cpu().numpy()
                        if kpts.shape[0] >= 17:
                            score = ready_score_from_kpts(kpts)
                if label:
                    cv2.putText(frame, label, (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                            (255, 255, 255), 2, cv2.LINE_AA)

                if tgt_bbox is not None:
                    x1, y1, x2, y2 = map(int, tgt_bbox)

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(frame, "TARGET", (x1, max(0, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                            (0, 255, 0), 2, cv2.LINE_AA)

                txt = f"ReadyScore: {score:5.1f}" if (score is not None and np.
            isfinite(score)) else "ReadyScore: n/a"
                cv2.putText(frame, txt, (20, 85),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                        (255, 255, 255), 2, cv2.LINE_AA)
                writer.write(frame)











                writer.release()
                return True

# Defines a helper that renders model outputs into a human readable video artifact for review
        def convert_overlay_to_h264_faststart(inp, outp):
                if os.path.exists(outp):
                os.remove(outp)
                cmd = [
                "ffmpeg", "-hide_banner", "-y",
                "-i", inp,
                "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-preset", "veryfast", "-crf", "23",
                "-movflags", "+faststart",
                "-an",
                outp
                ]

                run_cmd(cmd)

        import os

        if "RUN_DIR" not in globals():
                RUN_DIR = "runs_ready_score"

        os.makedirs(RUN_DIR, exist_ok=True)

        AM_OVERLAY_RAW = os.path.join(RUN_DIR, "amateur_clip1_overlay_raw.mp4")
        AM_OVERLAY_RAW_2 = os.path.join(RUN_DIR, "amateur_clip2_overlay_raw.mp4")
        AM_OVERLAY_RAW_3 = os.path.join(RUN_DIR, "amateur_clip3_overlay_raw.mp4")

        PRO_OVERLAY_RAW = os.path.join(RUN_DIR, "pro_clip1_overlay_raw.mp4")
        PRO_OVERLAY_RAW_2 = os.path.join(RUN_DIR, "pro_clip2_overlay_raw.mp4")
        PRO_OVERLAY_RAW_3 = os.path.join(RUN_DIR, "pro_clip3_overlay_raw.mp4")

        AM_OVERLAY_WEB = os.path.join(RUN_DIR, "amateur_clip1_overlay_h264.mp4")
        AM_OVERLAY_WEB_2 = os.path.join(RUN_DIR, "amateur_clip2_overlay_h264.mp4")

        AM_OVERLAY_WEB_3 = os.path.join(RUN_DIR, "amateur_clip3_overlay_h264.mp4")

        PRO_OVERLAY_WEB = os.path.join(RUN_DIR, "pro_clip1_overlay_h264.mp4")
        PRO_OVERLAY_WEB_2 = os.path.join(RUN_DIR, "pro_clip2_overlay_h264.mp4")
        PRO_OVERLAY_WEB_3 = os.path.join(RUN_DIR, "pro_clip3_overlay_h264.mp4")











        print("Overlay path variables defined.")

        Overlay path variables defined.

In [ ]:
write_overlay_raw_locked(AM_CLIP, AM_OVERLAY_RAW, model, AM_TARGET_BBOX, conf=CONF, label="AMATEUR 1 (10s)")
write_overlay_raw_locked(AM_CLIP_2, AM_OVERLAY_RAW_2, model, AM_TARGET_BBOX_2, conf=CONF, label="AMATEUR 2 (10s)")
write_overlay_raw_locked(AM_CLIP_3, AM_OVERLAY_RAW_3, model, AM_TARGET_BBOX_3, conf=CONF, label="AMATEUR 3 (10s)")
write_overlay_raw_locked(PRO_CLIP, PRO_OVERLAY_RAW, model, PRO_TARGET_BBOX, conf=CONF, label="PRO 1 (10s)")
write_overlay_raw_locked(PRO_CLIP_2, PRO_OVERLAY_RAW_2, model, PRO_TARGET_BBOX_2, conf=CONF, label="PRO 2 (10s)")
write_overlay_raw_locked(PRO_CLIP_3, PRO_OVERLAY_RAW_3, model, PRO_TARGET_BBOX_3, conf=CONF, label="PRO 3 (10s)")
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW, AM_OVERLAY_WEB)
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW_2, AM_OVERLAY_WEB_2)
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW_3, AM_OVERLAY_WEB_3)
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW, PRO_OVERLAY_WEB)
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW_2, PRO_OVERLAY_WEB_2)
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW_3, PRO_OVERLAY_WEB_3)
print("Overlays created:")
print(AM_OVERLAY_WEB, AM_OVERLAY_WEB_2, AM_OVERLAY_WEB_3)
print(PRO_OVERLAY_WEB, PRO_OVERLAY_WEB_2, PRO_OVERLAY_WEB_3)
Overlays created:
runs_ready_score/amateur_clip1_overlay_h264.mp4
runs_ready_score/amateur_clip2_overlay_h264.mp4
runs_ready_score/amateur_clip3_overlay_h264.mp4
runs_ready_score/pro_clip1_overlay_h264.mp4
runs_ready_score/pro_clip2_overlay_h264.mp4
runs_ready_score/pro_clip3_overlay_h264.mp4

In [ ]:
# Defines _video_html to encapsulate reusable logic within the video analytics pipeline
        def _video_html(path, width=520):
                if not os.path.exists(path):
                return f"<div style='color:#b00'>Missing: {path}</div>"
                return Video(path, embed=True, width=width)._repr_html_()

# Defines a helper that renders model outputs into a human readable video artifact for review
        def show_overlay_pairs():










                pairs = [
                ("Pair 1 (10s)", AM_OVERLAY_WEB, PRO_OVERLAY_WEB),
                ("Pair 2 (10s)", AM_OVERLAY_WEB_2, PRO_OVERLAY_WEB_2),
                ("Pair 3 (10s)", AM_OVERLAY_WEB_3, PRO_OVERLAY_WEB_3),
                ]
                for title, am_path, pro_path in pairs:
                html = f"""
                <div style="margin: 10px 0 22px 0;">
                <div style="font-weight:700; font-size:16px; margin-bottom:6px;
            ">{title}</div>
                <div style="display:flex; gap:18px; align-items:flex-start; flex-wrap:
            wrap;">
                    <div style="min-width:540px;">
                        <div style="font-weight:600; margin:0 0 6px 0;">Amateur</div>
                        {_video_html(am_path, width=520)}
                    </div>
                    <div style="min-width:540px;">
                        <div style="font-weight:600; margin:0 0 6px 0;">Pro</div>
                        {_video_html(pro_path, width=520)}
                    </div>
                </div>
                </div>
                """
                display(HTML(html))
        show_overlay_pairs()

        <IPython.core.display.HTML object>
        <IPython.core.display.HTML object>

        <IPython.core.display.HTML object>

In [ ]:
# Defines a helper that formats results into a constrained summary suitable for downstream reporting
def _safe_locked_summary(path, bbox):
    if not path or not os.path.exists(path):
        return {"frames_scored": 0, "mean": None, "p10": None, "p50": None, "p90": None, "target_id": None}
    return summarize_video_ready_score_locked(path, model, bbox, conf=CONF)
am_summary_1 = _safe_locked_summary(AM_CLIP, AM_TARGET_BBOX)
am_summary_2 = _safe_locked_summary(AM_CLIP_2, AM_TARGET_BBOX_2)
am_summary_3 = _safe_locked_summary(AM_CLIP_3, AM_TARGET_BBOX_3)

pro_summary_1 = _safe_locked_summary(PRO_CLIP, PRO_TARGET_BBOX)
pro_summary_2 = _safe_locked_summary(PRO_CLIP_2, PRO_TARGET_BBOX_2)
pro_summary_3 = _safe_locked_summary(PRO_CLIP_3, PRO_TARGET_BBOX_3)












rows = [
    {"clip_type":"amateur","clip_id":1,"clip_path":AM_CLIP, **am_summary_1},
    {"clip_type":"amateur","clip_id":2,"clip_path":AM_CLIP_2, **am_summary_2},
    {"clip_type":"amateur","clip_id":3,"clip_path":AM_CLIP_3, **am_summary_3},
    {"clip_type":"pro","clip_id":1,"clip_path":PRO_CLIP, **pro_summary_1},
    {"clip_type":"pro","clip_id":2,"clip_path":PRO_CLIP_2, **pro_summary_2},
    {"clip_type":"pro","clip_id":3,"clip_path":PRO_CLIP_3, **pro_summary_3},
]

df = pd.DataFrame(rows)
df = df[["clip_type","clip_id","clip_path","target_id","frames_scored","mean","p10","p50","p90"]].
sort_values(["clip_type","clip_id"]).reset_index(drop=True)
display(df)
grouped = df.groupby("clip_type")[["mean","p10","p50","p90","frames_scored"]].
agg(["mean","std","min","max"])
print("\nGrouped summary (ReadyScore: higher is better)")
display(grouped)
clip_type clip_id                  clip_path target_id \
0  amateur     1 runs_ready_score/amateur_clip1.mp4 466
1  amateur     2 runs_ready_score/amateur_clip2.mp4 549
2  amateur     3 runs_ready_score/amateur_clip3.mp4 581
3     pro      1    runs_ready_score/pro_clip1.mp4 655
4     pro      2    runs_ready_score/pro_clip2.mp4 687
5     pro      3    runs_ready_score/pro_clip3.mp4 800

frames_scored   mean      p10     p50      p90
0         282 60.204179 44.670534 61.810236 73.938388
1         193 70.079891 55.870137 70.428532 85.795101
2         263 57.275512 38.996841 58.315186 74.733034
3         205 70.104989 53.488295 71.772493 83.182794
4         300 61.425632 44.449437 62.898565 76.827614
5         177 72.373900 56.613542 73.624761 87.981999


Grouped summary (ReadyScore: higher is better)
            mean                              p10         \
            mean    std      min      max     mean    std

clip_type
amateur  62.519861 6.708936 57.275512 70.079891 46.512504 8.586132
pro      67.968174 5.778463 61.425632 72.373900 51.517091 6.317088

                                p50                          \
            min      max     mean     std     min      max










clip_type
amateur  38.996841 55.870137 63.517985 6.234629 58.315186 70.428532
pro      44.449437 56.613542 69.431940 5.733364 62.898565 73.624761

            p90                          frames_scored       \
            mean    std      min      max       mean     std
clip_type
amateur  78.155508 6.628001 73.938388 85.795101 246.000000 46.872167
pro      82.664136 5.595251 76.827614 87.981999 227.333333 64.469631


        min max
clip_type
amateur  193 282
pro      177 300

In [ ]:
md = """
**Interpreting the summary stats (ReadyScore):**
- **Higher is better** (100 = excellent “ready position”, 0 = poor readiness).
- **p90** = the player’s **best** readiness moments (top 10% of frames).
- **p50** = the typical / median frame.
- **p10** = the player’s **worst** readiness moments (bottom 10% of frames).
"""
display(HTML(md))

<IPython.core.display.HTML object>

In [ ]:
# Imports libraries for data engineering, video processing, and model inference
from openai import OpenAI

# os.environ["OPENAI_API_KEY"] =
# Initializes the OpenAI client to support downstream LLM scoring and narrative interpretation
client = OpenAI()
pro_ref = grouped.loc["pro"]
pro_reference = {
    "mean_ready_score": float(pro_ref[("mean", "mean")]),
    "median_ready_score": float(pro_ref[("p50", "mean")]),
    "low_end_ready_score": float(pro_ref[("p10", "mean")]), # worse tail










    "high_end_ready_score": float(pro_ref[("p90", "mean")]), # best tail
}
print("Pro reference:", pro_reference)

# Defines generate_feedback to encapsulate reusable logic within the video analytics pipeline
def generate_feedback(amateur_row, pro_ref):
# Defines fmt to encapsulate reusable logic within the video analytics pipeline
    def fmt(x):
        try:
        if x is None:
            return "N/A"
        x = float(x)
        if not np.isfinite(x):
            return "N/A"
        return f"{x:.1f}"
        except Exception:
        return "N/A"


    prompt = f"""
You are a pickleball coach providing technique feedback based on pose-derived metrics.
The metrics summarize a player's ready-position posture over a 10-second clip.
ReadyScore ranges 0–100, and HIGHER is BETTER (100 = excellent readiness).

Professional reference (average across clips):
- Mean ready score: {fmt(pro_ref['mean_ready_score'])}
- Median ready score: {fmt(pro_ref['median_ready_score'])}
- Low-end (p10, worse moments): {fmt(pro_ref['low_end_ready_score'])}
- High-end (p90, best moments): {fmt(pro_ref['high_end_ready_score'])}

Amateur clip metrics:
- Mean ready score: {fmt(amateur_row.get('mean'))}
- Median ready score: {fmt(amateur_row.get('p50'))}
- Low-end (p10, worse moments): {fmt(amateur_row.get('p10'))}
- High-end (p90, best moments): {fmt(amateur_row.get('p90'))}

Task:
1) Compare the amateur to the professional reference.
2) Identify 2–3 concrete posture or readiness issues (knee bend, stance width, 
forward lean, paddle/hand position).
3) Include one positive observation.
4) Suggest one actionable drill.
5) Keep feedback concise, coach-like, and specific.
"""
    resp = client.chat.completions.create(
# Selects GPT 4o mini to minimize cost and latency while supporting structured analysis over extracted pose features
        model="gpt-4o-mini",
        messages=[










        {"role": "system", "content": "You are an expert pickleball coach.
"},
        {"role": "user", "content": prompt},
        ],
        temperature=0.4,
    )
    return resp.choices[0].message.content.strip()
amateur_feedback = {}
for _, row in df[df["clip_type"] == "amateur"].iterrows():
    clip_id = row["clip_id"]
    amateur_feedback[f"amateur_clip_{clip_id}"] = generate_feedback(row, pro_reference)

for clip, text in amateur_feedback.items():
    print("\n" + "=" * 60)
    print(clip.upper())
    print(text)

Pro reference: {'mean_ready_score': 67.9681739277063, 'median_ready_score':
69.43193982633028, 'low_end_ready_score': 51.517091127633364,
'high_end_ready_score': 82.6641356294773}

============================================================
AMATEUR_CLIP_1

**1) Comparison:**
- Your mean ready score of **60.2** and median of **61.8** are below the
professional averages (68.0 and 69.4, respectively). This indicates that your
readiness in posture needs improvement.
- The low-end score of **44.7** suggests that there are moments where your
posture is significantly lacking compared to the professional standard.

**2) Posture/Readiness Issues:**
- **Knee Bend:** Your knee bend appears insufficient, which can limit your
ability to react quickly.
- **Stance Width:** Your stance may be too narrow, affecting your balance and
stability during play.
- **Paddle Position:** The paddle is often too low or not in a ready position,
which delays your response time to incoming shots.


**3) Positive Observation:**
- Your overall body alignment is good; you maintain a straight back, which is
crucial for effective movement.

**4) Actionable Drill:**










- **Ready Position Drill:** Practice the "Ready Position Shuffle." Start in your
ready position with knees bent, feet shoulder-width apart, and paddle at waist
height. Shuffle side to side while maintaining this posture for 30 seconds.
Focus on keeping your knees bent and stance wide. Repeat this drill several
times to reinforce muscle memory.

Stay focused on these areas, and you'll see improvement in your readiness on the
court!

============================================================
AMATEUR_CLIP_2

1. **Comparison to Professional Reference:**
- Your mean ready score of **70.1** is slightly above the professional
average of **68.0**, indicating a solid foundation in your readiness. However,
your median score of **70.4** is also higher than the professional median of
**69.4**, showing consistency in your readiness posture.


2. **Posture/Readiness Issues:**
- **Knee Bend:** Your knee bend appears insufficient at times, which can
hinder your ability to react quickly. Aim for a deeper bend to enhance agility.
- **Stance Width:** Your stance width is inconsistent; at times, it’s too
narrow, affecting your balance. A wider base will improve stability and
readiness.
- **Paddle Position:** Your paddle is often held too low, which can delay
your reaction time. Keep your paddle up and ready for quicker responses.

3. **Positive Observation:**
- Your forward lean is commendable, as it shows you are engaged and ready to
move toward the ball. This posture is crucial for effective court coverage.

4. **Actionable Drill:**
- **Ready Position Drill:** Practice the "Ready Position Flow" drill. Stand
in your ready position and focus on maintaining a deep knee bend and wide
stance. Alternate between moving side to side while keeping your paddle up and
ready. Do this for 5 minutes, emphasizing fluid transitions and maintaining
posture.

Keep up the good work, and focus on these areas to elevate your game!

============================================================

AMATEUR_CLIP_3

1. **Comparison to Professional Reference:**
- Your mean ready score of **57.3** is significantly below the professional
average of **68.0**, indicating room for improvement in your readiness.









- The median score of **58.3** also falls short of the professional median of
**69.4**, suggesting consistent issues in your posture.

2. **Posture/Readiness Issues:**
- **Knee Bend:** Your knee bend appears insufficient, which can limit your
ability to react quickly. Aim for a deeper bend to enhance your stability and
readiness.
- **Stance Width:** Your stance seems too narrow, reducing your balance. A
wider stance will provide better support and allow for quicker lateral
movements.
- **Forward Lean:** There is minimal forward lean in your posture. A slight
forward lean will help you stay engaged and ready to move in any direction.

3. **Positive Observation:**
- Your paddle position is generally good, staying close to your body and
ready for quick shots. This is a strong foundation to build upon.

4. **Actionable Drill:**
- **Ready Position Drill:** Practice the "Ready Position Hold" drill. Stand

in your ready position with a deep knee bend and wide stance. Hold this position
for 30 seconds while focusing on maintaining a forward lean and keeping your
paddle up. Repeat this 5 times, gradually increasing the duration as you become
more comfortable.

Focus on these adjustments, and you’ll see improvements in your readiness on the
court! Keep up the hard work!

        [ ]: